# 🚀 Тестирование Agent Framework

Этот ноутбук пошагово проверяет все компоненты фреймворка агентов:

1. **Тест базовых функций** - проверка инструментов
2. **Test Agent** - простейший агент (1 состояние)
3. **My Agent** - переходы между состояниями (2 состояния)
4. **Router Agent** - условный роутинг (ветвление)
5. **Multi-Agent System** - один агент вызывает другого
6. Проверка **сессий**
7. Проверка **UI**
Бэкенд (GigaChat или LM Studio) настраивается в `config.yaml`.

In [1]:
# Установка зависимостей (запустите один раз)
%uv pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Using Python 3.12.12 environment at: C:\Users\accordij\Anaconda3\envs\agent_cdo
Resolved 87 packages in 1.45s
Prepared 1 package in 292ms
Uninstalled 6 packages in 41ms
Installed 3 packages in 40ms
 - gigachat==0.1.43
 ~ gigachat==0.2.0
 - langgraph-prebuilt==1.0.1
 ~ langgraph-prebuilt==1.0.9
 - langgraph-sdk==0.2.15
 - langgraph-sdk==0.3.12
 + langgraph-sdk==0.3.13


## 📦 Общая настройка

Импорты и конфигурация для всех тестов.

In [2]:
# Импорты
import yaml

# Импорты агентов
from src.agents.test_agent import TestAgent
from src.agents.my_agent import MyAgent
from src.agents.router_agent import RouterAgent
from src.agents.supervisor_agent import SupervisorAgent

# Импорты инструментов и клиентов
from src.tools.tools import (
    get_tools_dict,
    reset_memory,
    calculator,
)
from src.connections.clients import get_llm_client

# Logging (настройка уровня в config.yaml -> logging.level)
from src.agent_engine import init_logging

print("✓ Импорты выполнены успешно")

# Загрузка конфигурации
with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

backend = config['active_backend']
recursion_limit = config['agent']['recursion_limit']

print(f"Активный бэкенд: {backend}")
print(f"Лимит рекурсии графа: {recursion_limit}")

# Создание LLM клиента
llm = get_llm_client(backend, config)
print(f"✓ LLM клиент создан: {type(llm).__name__}")

# Инициализация logging (уровень задаётся в config.yaml)
init_logging()
print("✓ Logging инициализирован")

✓ Импорты выполнены успешно
Активный бэкенд: lmstudio
Лимит рекурсии графа: 30
✓ LLM клиент создан: ChatOpenAI
logging: level=detailed
✓ Logging инициализирован


In [3]:
# === Transition behavior probe (copy-paste cell) ===
import time
import json
import re
from collections import Counter

from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.prebuilt import create_react_agent

def make_transition_tool(require_summary: bool, require_reasoning: bool):
    allowed = {"stay", "END", "work", "summarize", "math", "text", "error"}

    @tool
    def transition(next_state: str, summary: str = "", reasoning: str = "") -> str:
        """Переход в другое состояние.
        Всегда вызови этот инструмент, когда задача завершена.
        """
        if next_state not in allowed:
            return f"ERR: invalid next_state='{next_state}'. allowed={sorted(allowed)}"
        if require_summary and not (summary or "").strip():
            return "ERR: summary is required"
        if require_reasoning and not (reasoning or "").strip():
            return "ERR: reasoning is required"
        return f"OK: transition accepted -> {next_state}"

    return transition

def extract_transition_signal(messages):
    # 1) canonical tool_call
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                if tc.get("name") == "transition":
                    args = tc.get("args", {}) or {}
                    return ("tool_call", args)

    # 2) text fallback
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content:
            txt = m.content
            mt = re.search(r'"next_state"\s*:\s*"([^"]+)"', txt)
            if mt:
                ns = mt.group(1)
                sm = re.search(r'"summary"\s*:\s*"([^"]*)"', txt)
                rs = re.search(r'"reasoning"\s*:\s*"([^"]*)"', txt)
                return ("text_json", {
                    "next_state": ns,
                    "summary": sm.group(1) if sm else "",
                    "reasoning": rs.group(1) if rs else "",
                })

    return ("none", {})

def run_probe(mode_name: str, require_summary: bool, require_reasoning: bool, trials: int = 8):
    transition_tool = make_transition_tool(require_summary, require_reasoning)
    agent = create_react_agent(llm, [transition_tool])  # llm должен быть уже инициализирован в ноутбуке

    stats = Counter()
    latencies = []
    samples = []

    # Режим-инструкция
    if mode_name == "full":
        task = (
            "Задача завершена. ОБЯЗАТЕЛЬНО вызови transition с полями: "
            "next_state='END', summary='...', reasoning='...'. "
            "После вызова больше не вызывай инструменты."
        )
    else:
        task = (
            "Задача завершена. ОБЯЗАТЕЛЬНО вызови transition только с next_state='END'. "
            "summary/reasoning можно оставить пустыми. "
            "После вызова больше не вызывай инструменты."
        )

    for i in range(trials):
        t0 = time.perf_counter()
        res = agent.invoke({"messages": [HumanMessage(content=task)]}, config={"recursion_limit": 12})
        dt = time.perf_counter() - t0
        latencies.append(dt)

        signal_type, payload = extract_transition_signal(res["messages"])
        stats[signal_type] += 1

        # сколько tool_calls было вообще
        tool_calls_total = 0
        for m in res["messages"]:
            if isinstance(m, AIMessage) and m.tool_calls:
                tool_calls_total += len(m.tool_calls)

        samples.append({
            "trial": i + 1,
            "signal": signal_type,
            "payload": payload,
            "tool_calls_total": tool_calls_total,
            "latency_s": round(dt, 3),
            "last_ai": next(
                (m.content for m in reversed(res["messages"])
                 if isinstance(m, AIMessage) and (m.content or "").strip()),
                ""
            )[:220]
        })

    print(f"\n=== MODE: {mode_name} | trials={trials} ===")
    print("stats:", dict(stats))
    print("avg_latency_s:", round(sum(latencies) / len(latencies), 3))
    print("p95_latency_s:", round(sorted(latencies)[max(0, int(0.95 * len(latencies)) - 1)], 3))
    print("\nSample[0]:")
    print(json.dumps(samples[0], ensure_ascii=False, indent=2))
    return {"mode": mode_name, "stats": dict(stats), "latencies": latencies, "samples": samples}

full_result = run_probe("full", require_summary=True, require_reasoning=True, trials=8)
fast_result = run_probe("fast", require_summary=False, require_reasoning=False, trials=8)

print("\n=== Comparison ===")
print("full avg:", round(sum(full_result["latencies"]) / len(full_result["latencies"]), 3))
print("fast avg:", round(sum(fast_result["latencies"]) / len(fast_result["latencies"]), 3))
print("full stats:", full_result["stats"])
print("fast stats:", fast_result["stats"])

C:\Users\accordij\AppData\Local\Temp\ipykernel_21840\527769174.py:57: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [transition_tool])  # llm должен быть уже инициализирован в ноутбуке



=== MODE: full | trials=8 ===
stats: {'tool_call': 8}
avg_latency_s: 3.713
p95_latency_s: 3.534

Sample[0]:
{
  "trial": 1,
  "signal": "tool_call",
  "payload": {
    "next_state": "END",
    "summary": "Task completed successfully.",
    "reasoning": "All requirements met."
  },
  "tool_calls_total": 1,
  "latency_s": 6.675,
  "last_ai": "**END**"
}


C:\Users\accordij\AppData\Local\Temp\ipykernel_21840\527769174.py:57: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [transition_tool])  # llm должен быть уже инициализирован в ноутбуке



=== MODE: fast | trials=8 ===
stats: {'tool_call': 8}
avg_latency_s: 2.673
p95_latency_s: 2.759

Sample[0]:
{
  "trial": 1,
  "signal": "tool_call",
  "payload": {
    "next_state": "END"
  },
  "tool_calls_total": 1,
  "latency_s": 3.105,
  "last_ai": "(Transition complete.)"
}

=== Comparison ===
full avg: 3.713
fast avg: 2.673
full stats: {'tool_call': 8}
fast stats: {'tool_call': 8}


In [4]:
# === Auto-transition + early-break probe ===
import time
import statistics
from collections import Counter

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# llm должен быть уже инициализирован в ноутбуке (как у тебя выше)

class EarlyTransition(Exception):
    """Сигнал раннего выхода после авто-перехода."""
    pass

def run_auto_transition_probe(trials=8, hard_break=False):
    runtime = {
        "transition": None,   # {"next_state": "...", "summary": "...", "source": "tool_auto"}
    }

    @tool
    def auto_route(result_type: str, summary: str = "") -> str:
        """
        Авто-переход без явного transition tool.
        Используй этот инструмент, когда уже понятно, куда переходить.
        Args:
            result_type: 'ok' или 'need_clarify'
            summary: краткий контекст для следующего состояния (может быть пустым)
        """
        if result_type == "ok":
            nxt = "END"
        else:
            nxt = "clarify"

        runtime["transition"] = {
            "next_state": nxt,
            "summary": summary or "",
            "source": "tool_auto",
        }

        if hard_break:
            # Имитируем "жесткий ранний обрыв" цепочки после фиксации перехода
            raise EarlyTransition(f"EARLY_BREAK -> {nxt}")

        return f"AUTO_ROUTE_OK -> {nxt}"

    # Можно добавить обычный transition tool для сравнения (необязательно)
    @tool
    def transition(next_state: str, summary: str = "") -> str:
        """Обычный переход."""
        runtime["transition"] = {
            "next_state": next_state,
            "summary": summary or "",
            "source": "transition_tool",
        }
        return f"TRANSITION_OK -> {next_state}"

    agent = create_react_agent(llm, [auto_route, transition])

    mode = "HARD_BREAK" if hard_break else "SOFT"
    stats = Counter()
    latencies = []
    samples = []

    prompt = (
        "Сделай ровно один вызов инструмента auto_route. "
        "Передай result_type='ok' и summary='готово'. "
        "После вызова не вызывай другие инструменты."
    )

    for i in range(trials):
        runtime["transition"] = None
        t0 = time.perf_counter()

        try:
            res = agent.invoke(
                {"messages": [HumanMessage(content=prompt)]},
                config={"recursion_limit": 10},
            )
            dt = time.perf_counter() - t0
            latencies.append(dt)

            if runtime["transition"]:
                stats["transition_captured"] += 1
            else:
                stats["no_transition"] += 1

            stats["invoke_returned"] += 1

            samples.append({
                "trial": i + 1,
                "invoke": "returned",
                "transition": runtime["transition"],
                "latency_s": round(dt, 3),
                "messages_count": len(res.get("messages", [])),
            })

        except EarlyTransition as e:
            dt = time.perf_counter() - t0
            latencies.append(dt)

            # Ключевая проверка: сохранился ли переход ДО обрыва?
            if runtime["transition"]:
                stats["transition_captured"] += 1
            else:
                stats["no_transition"] += 1

            stats["early_break_exception"] += 1

            samples.append({
                "trial": i + 1,
                "invoke": "early_break_exception",
                "exception": str(e),
                "transition": runtime["transition"],
                "latency_s": round(dt, 3),
            })

        except Exception as e:
            dt = time.perf_counter() - t0
            latencies.append(dt)
            stats["other_exception"] += 1
            samples.append({
                "trial": i + 1,
                "invoke": "other_exception",
                "exception": repr(e),
                "transition": runtime["transition"],
                "latency_s": round(dt, 3),
            })

    print(f"\n=== AUTO-TRANSITION PROBE: {mode} | trials={trials} ===")
    print("stats:", dict(stats))
    print("avg_latency_s:", round(statistics.mean(latencies), 3))
    print("p95_latency_s:", round(sorted(latencies)[max(0, int(0.95 * len(latencies)) - 1)], 3))
    print("sample[0]:", samples[0])

    return {"mode": mode, "stats": dict(stats), "samples": samples, "latencies": latencies}

soft_res = run_auto_transition_probe(trials=8, hard_break=False)
hard_res = run_auto_transition_probe(trials=8, hard_break=True)

print("\n=== COMPARISON ===")
print("soft stats:", soft_res["stats"])
print("hard stats:", hard_res["stats"])
print("soft avg:", round(statistics.mean(soft_res["latencies"]), 3))
print("hard avg:", round(statistics.mean(hard_res["latencies"]), 3))

C:\Users\accordij\AppData\Local\Temp\ipykernel_21840\2568068963.py:58: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [auto_route, transition])



=== AUTO-TRANSITION PROBE: SOFT | trials=8 ===
stats: {'transition_captured': 8, 'invoke_returned': 8}
avg_latency_s: 4.124
p95_latency_s: 3.901
sample[0]: {'trial': 1, 'invoke': 'returned', 'transition': {'next_state': 'END', 'summary': 'готово', 'source': 'tool_auto'}, 'latency_s': 7.183, 'messages_count': 4}


C:\Users\accordij\AppData\Local\Temp\ipykernel_21840\2568068963.py:58: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [auto_route, transition])



=== AUTO-TRANSITION PROBE: HARD_BREAK | trials=8 ===
stats: {'transition_captured': 8, 'early_break_exception': 8}
avg_latency_s: 1.98
p95_latency_s: 2.402
sample[0]: {'trial': 1, 'invoke': 'early_break_exception', 'exception': 'EARLY_BREAK -> END', 'transition': {'next_state': 'END', 'summary': 'готово', 'source': 'tool_auto'}, 'latency_s': 1.849}

=== COMPARISON ===
soft stats: {'transition_captured': 8, 'invoke_returned': 8}
hard stats: {'transition_captured': 8, 'early_break_exception': 8}
soft avg: 4.124
hard avg: 1.98


---

## 1️⃣ Тест базовых функций

Проверяем что инструменты работают корректно.

In [ ]:
# print("=" * 60)
# print("ТЕСТ 1: Базовые функции")
# print("=" * 60)

# # Тест калькулятора
# print("\n1. Тест калькулятора:")
# result = calculator.invoke("2 ** 10")
# print(f"   2^10 = {result}")
# assert result == "1024", "Калькулятор работает неправильно!"
# print("   ✓ Калькулятор работает")

# # Тест памяти
# print("\n2. Тест памяти:")
# from src.tools.tools import memory
# reset_memory()
# memory.invoke({"action": "save", "key": "test_key", "value": "test_value"})
# result = memory.invoke({"action": "get", "key": "test_key"})
# assert result.get("ok") and result.get("entries", {}).get("test_key") == "test_value", (
#     "Память работает неправильно!"
# )
# print("   ✓ Память работает")

# print("\n✅ Все базовые функции работают корректно!")

ТЕСТ 1: Базовые функции

1. Тест калькулятора:
   2^10 = 1024
   ✓ Калькулятор работает

2. Тест памяти:
   ✓ Память работает

✅ Все базовые функции работают корректно!


In [3]:
# print("=" * 60)
# print("ТЕСТ 1.5: Проверка подключения к LLM")
# print("=" * 60)

# from langchain_core.messages import HumanMessage

# try:
#     test_response = llm.invoke([HumanMessage(content="Ответь одним словом: 2+2=")])
#     print(f"\n✓ LLM ответил: {test_response.content.strip()}")
#     print("✓ Подключение к LLM работает")
# except Exception as e:
#     print(f"\n✗ Ошибка подключения к LLM: {e}")
#     print("Проверьте что LM Studio запущен и доступен на порту из config.yaml")

ТЕСТ 1.5: Проверка подключения к LLM

✓ LLM ответил: Четыре
✓ Подключение к LLM работает


---

## 2️⃣ Test Agent - Простейший агент

Граф: `[work] → END`

Проверяем:
- Создание агента через AgentConfig
- Работу с инструментами
- Переход в END по ключевому слову

In [4]:
print("======== ТЕСТ 2: Test Agent (1 состояние) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="test_agent")

# Создание агента
test_agent = TestAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {test_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(test_agent.visualize())

======== ТЕСТ 2: Test Agent (1 состояние) ========

✓ Агент создан: TestAgent(id=TestAgent_1325294517952, states=1, sessions=enabled)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 1
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Единственное рабочее состояние
    Переходы: END
    Инструменты (shared): calculator, memory, think
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [5]:
result = test_agent.invoke([
    "Посчитай 15 * 23 и сохрани результат в память с ключом 'result'"
])

RUN b754aa6a | TestAgent_1325294517952 | started
STATE START -> work
  REQ 1 | msgs=2 in=701 out=37
    [SYS] Ты простой тестовый агент.

Твои возможности:
- calculator: вычисление математических выражений
- memory: сохранение/чтение данных
- think: размышления

Выполни запрос пользователя, используя доступные инструменты.


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему переходишь именно туда.
- summary: что было сделано и что нужно сделать в следующем состоянии. Укажи какие ключи сохранены в memory.
- next_state: одно из доступных значений:
  - "stay" — остаться в текущем состоянии (нужно ещё поработать)
  - "END"

ВАЖНО: после вызова transition не вызывай другие инструменты — напиши финальный ответ.
    [USER] Посчитай 15 * 23 и сохрани результат в память с ключом 'result'
  TOOL calculator params={'expression': '15 * 23'}
  TOOL memory params={'action': 'save', 'key': 'calculator_last', 'value': ' 15 * 23 

---

## 3️⃣ My Agent - Переходы между состояниями

Граф: `[work] ⟳ → [summarize] → END`

Проверяем:
- Циклическое состояние (work возвращается в себя)
- Условный переход по ключевому слову
- Работу с памятью между состояниями

In [4]:
print("======== ТЕСТ 3: My Agent (2 состояния + переход) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="my_agent")

# Создание агента
my_agent = MyAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {my_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(my_agent.visualize())

======== ТЕСТ 3: My Agent (2 состояния + переход) ========

✓ Агент создан: MyAgent(id=MyAgent_1911739945648, states=2, sessions=enabled)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: work
  - Состояний: 2
  - Уникальных тулов: 6
  - Ключей в памяти сейчас: 0

Состояния:
  - work (entry)
    Описание: Основное рабочее состояние с полным набором инструментов
    Переходы: summarize
    Инструменты (shared): calculator, ask_human, think, memory
    Инструменты (local): plot_chart
    Memory injections: нет
  - summarize
    Описание: Финальное состояние для рефлексии и подведения итогов
    Переходы: END
    Инструменты (shared): summarize, memory
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [5]:
# Запуск агента
# Для теста картинок в UI:
# Нарисуй столбчатый график продаж по кварталам: Q1=120, Q2=95, Q3=140, Q4=200
result = my_agent.invoke([
    "давай подсчитаем площадь круга?"
])

RUN 9834376b | MyAgent_1911739945648 | started
STATE START -> work
  REQ 1 | msgs=2 in=1129 out=37
    [SYS] Ты полезный помощник для математических вычислений и визуализации данных.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное
- plot_chart: построить график (bar/line/pie) — он автоматически появится у пользователя

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Если результаты стоит визуализировать — построй график через plot_chart
6. Сохрани важные результаты через memory
7. Вызови transition(next_state="summarize") — это обязательный последний шаг


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вы

---

## 4️⃣ Router Agent - Условный роутинг

Граф: `[classify] → [math | text | error] → END`

Проверяем:
- Классификацию запроса
- Роутер с несколькими выходами
- Разные пути обработки

In [ ]:
print("======== ТЕСТ 4: Router Agent (условный роутинг) ========")

# Подготовка
reset_memory()
tools_dict = get_tools_dict(agent_name="router_agent")

# Создание агента
router_agent = RouterAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {router_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(router_agent.visualize())

ТЕСТ 4: Router Agent (условный роутинг)

✓ Агент создан: RouterAgent(id=RouterAgent_2143587284992, states=4)

Структура графа:
Граф агента (preflight):

Сводка:
  - Entry point: classify
  - Состояний: 4
  - Уникальных тулов: 3
  - Ключей в памяти сейчас: 0

Состояния:
  - classify (entry)
    Описание: Классификация типа запроса
    Переходы: math, text, error
    Инструменты: think [shared], memory [shared]
    Memory injections: request_type
  - math
    Описание: Обработка математических запросов
    Переходы: END
    Инструменты: calculator [shared], memory [shared], think [shared]
    Memory injections: request_type, result
  - text
    Описание: Обработка текстовых запросов
    Переходы: END
    Инструменты: memory [shared], think [shared]
    Memory injections: request_type, response_text
  - error
    Описание: Обработка нераспознанных запросов
    Переходы: END
    Инструменты: think [shared]
    Memory injections: нет

Проверки конфигурации:
  ✓ Ошибок конфигурации не найден

In [ ]:
# Тест 1: Математический запрос
print("======== Тест 4.1: Математический запрос ========")

reset_memory()
result = router_agent.invoke(["Посчитай 5 + 5"])

In [ ]:
# Тест 2: Текстовый запрос
print("======== Тест 4.2: Текстовый запрос ========")

reset_memory()
result = router_agent.invoke(["Привет! Как дела?"])

---

## 5️⃣ Multi-Agent System - Композиция агентов

Граф: `Supervisor [delegate] → [aggregate] → END`
         ↓ вызывает ↓
       Test Agent, Router Agent

Проверяем:
- Регистрацию агентов в реестре
- Вызов агента из агента через инструмент call_agent
- Агрегацию результатов от нескольких агентов

In [6]:
print("======== ТЕСТ 5: Multi-Agent System (композиция агентов) ========")

# Подготовка
reset_memory()
print("\n✓ Подчиненные агенты будут зарегистрированы автоматически внутри SupervisorAgent")

# Создаем супервизора с инструментом call_agent
supervisor_tools_dict = get_tools_dict(agent_name="supervisor_agent")
supervisor = SupervisorAgent(llm, supervisor_tools_dict)

print(f"✓ Супервизор создан: {supervisor}")

# Визуализация графа
print("\nСтруктура графа супервизора:")
print(supervisor.visualize())

======== ТЕСТ 5: Multi-Agent System (композиция агентов) ========

✓ Подчиненные агенты будут зарегистрированы автоматически внутри SupervisorAgent
✓ Супервизор создан: SupervisorAgent(id=SupervisorAgent_1911739367840, states=2, sessions=enabled)

Структура графа супервизора:
Граф агента (preflight):

Сводка:
  - Entry point: delegate
  - Состояний: 2
  - Уникальных тулов: 4
  - Ключей в памяти сейчас: 0

Состояния:
  - delegate (entry)
    Описание: Делегирование задач специализированным агентам
    Переходы: aggregate
    Инструменты (shared): call_agent, memory, think
    Memory injections: request_type, response_text, delegation_notes
  - aggregate
    Описание: Агрегация результатов от агентов
    Переходы: END
    Инструменты (shared): memory, summarize, think
    Memory injections: request_type, response_text, delegation_notes

Проверки конфигурации:
  ✓ Ошибок конфигурации не найдено


In [7]:
# Запуск супервизора
print("\n" + "=" * 60)
print("Запуск: Посчитай 10 * 5 и поздоровайся")
print("=" * 60 + "\n")

result = supervisor.invoke([
    "Посчитай 10 * 5 через test_agent и передай приветствие через router_agent"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Multi-Agent System работает корректно!")


Запуск: Посчитай 10 * 5 и поздоровайся

RUN 8da6d659 | SupervisorAgent_1911739367840 | started
STATE START -> delegate
  REQ 1 | msgs=2 in=1191 out=54
    [SYS] Ты агент-супервизор, который координирует работу других агентов.

Твои возможности:
- call_agent: вызвать другого зарегистрированного агента
- memory: сохранить результаты работы агентов
- think: продумать стратегию делегирования

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Сначала проверь память: memory(action="list")
3. Определи режим работы:
   - Первый вход: если память пуста или нет ключа request_type
   - Повторный вход: если request_type уже есть
4. Для первого входа:
   - Выбери одного наиболее подходящего агента и вызови его через call_agent(agent_name="...", query="...")
   - Сохрани ключевой результат в память
5. Для повторного входа:
   - Сначала прочитай уже сохраненные ключи через memory(action="get", key="...")
   - НЕ вызывай повторно того же агента с тем же запросом, если в памяти уже есть достато

## Сессии и чекпоинты

Система сессий сохраняет полный `AgentState` (messages, memory, summary) после каждого шага агента в SQLite.

**Возможности:**
- Автоматическое создание сессии при каждом `invoke()`
- Восстановление после падения через `resume(session_id)`
- Именованные чекпоинты (защита от авто-удаления) через `tag_checkpoint()`
- Форк сессии с опциональными правками: `fork("name", edits={...})`
- Просмотр истории: `list_checkpoints(session_id)`

Настройки в `config.yaml` секция `sessions:`.

In [4]:
# ── Базовая работа с сессиями ──────────────────────────────────────────────
# Каждый invoke() автоматически создаёт новую сессию если session_id не передан.
# session_id сохраняется в agent.last_session_id

reset_memory()
tools_dict = get_tools_dict(agent_name="my_agent")
my_agent = MyAgent(llm, tools_dict)

# Запуск — новая сессия создаётся автоматически
result = my_agent.invoke(["Посчитай 2+2 и сохрани результат в память"])
session_id = my_agent.last_session_id
print(f"\nSession ID: {session_id}")

# Просмотр чекпоинтов сессии
print("\n── Чекпоинты сессии ──")
for cp in my_agent.list_checkpoints(session_id):
    name_tag = f" ★ {cp['name']}" if cp.get("is_named") else ""
    print(f"  {cp['state_name']:<20} {cp['timestamp'][:19]}{name_tag}")

# Тегировать последний чекпоинт
my_agent.tag_checkpoint(session_id, "after_calculation", note="результат вычислений")
print(f"\n✓ Чекпоинт 'after_calculation' сохранён")

# Посмотреть содержимое
state = my_agent.get_checkpoint_state("after_calculation")
print(f"memory: {state.get('memory', {})}")

RUN ec2e9df3 | MyAgent_2413891933280 | started


STATE START -> work
  REQ 1 | msgs=2 in=1132 out=40
    [SYS] Ты полезный помощник для математических вычислений и визуализации данных.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное
- plot_chart: построить график (bar/line/pie) — он автоматически появится у пользователя

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Если результаты стоит визуализировать — построй график через plot_chart
6. Сохрани важные результаты через memory
7. Вызови transition(next_state="summarize") — это обязательный последний шаг


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: почему

In [8]:
# ── Форк: запустить одну точку несколько раз с разными параметрами ─────────
# fork() создаёт новую независимую сессию из именованного чекпоинта.
# Оригинал не трогается — можно форкать сколько угодно раз.

print("── Форки из 'after_calculation' ──")

for extra_value in [10, 100]:
    reset_memory()
    new_sid = my_agent.fork(
        "after_calculation",
        edits={"memory": {"bonus": extra_value}},
        description=f"тест с bonus={extra_value}",
    )
    result = my_agent.invoke(
        [f"Сложи результат вычислений из памяти с bonus из памяти"],
        session_id=new_sid,
    )
    print(f"\n  bonus={extra_value} → session {new_sid[:8]}...")

# ── Продолжение существующей сессии ────────────────────────────────────────
# Передаём session_id — агент видит всю историю сообщений из этой сессии
print("\n── Продолжение сессии ──")
result = my_agent.invoke(
    ["Напомни какой был результат вычислений"],
    session_id=session_id,
)

# ── Список всех сессий ─────────────────────────────────────────────────────
print("\n── Все сессии my_agent ──")
for s in my_agent._sm.list_sessions("my_agent"):
    named_count = len(my_agent._sm.list_named_checkpoints(s["session_id"]))
    star = " ★" if named_count else ""
    print(f"  {s['session_id'][:8]}... | {s.get('description',''):<30} | {s.get('updated_at','')[:16]}{star}")

── Форки из 'after_calculation' ──
RUN f4f7a719 | MyAgent_2413891933280 | started


STATE summarize -> work
  REQ 1 | msgs=4 in=1172 out=63
    [SYS] Ты полезный помощник для математических вычислений и визуализации данных.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное
- plot_chart: построить график (bar/line/pie) — он автоматически появится у пользователя

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Если результаты стоит визуализировать — построй график через plot_chart
6. Сохрани важные результаты через memory
7. Вызови transition(next_state="summarize") — это обязательный последний шаг


## Переход в другое состояние
Когда задача в текущем состоянии выполнена, вызови инструмент transition:
- reasoning: по

In [ ]:
# ── Сценарий 1: КРОН / автоматический скрипт ──────────────────────────────
# resume() бросает NothingToResumeError если сессия уже завершилась (END).
# Крон ловит его — не тратит токены, не считает это ошибкой.

from src.agent_engine import NothingToResumeError
from src.tools.tools import reset_memory, _memory_store

print("── Краш-рекавери (cron-сценарий) ──")
reset_memory()
print(f"До: memory = {dict(_memory_store)}")

try:
    result = my_agent.resume(session_id)
    print("✓ Агент возобновлён и завершил работу")
except NothingToResumeError as e:
    print(f"ℹ {e}")
    print("  → Крон логирует и завершается без ошибки. Токены не потрачены.")

# ── Сценарий 2: только восстановить память (без перезапуска агента) ─────────
# Полезно после перезапуска ядра — просто загружаем память из БД.
print("\n── Восстановление памяти из последнего checkpoint ──")
memory = my_agent.restore_memory(session_id)
print(f"Память восстановлена: {memory}")

# ── Сценарий 3: интерактивный выбор checkpoint через виджет ────────────────
import ipywidgets as widgets
from IPython.display import display

checkpoints = my_agent.list_checkpoints(session_id)
if not checkpoints:
    print("Нет чекпоинтов для выбора")
else:
    def _cp_label(cp):
        name_tag = f" ★ {cp['name']}" if cp.get("name") else ""
        return f"{cp['state_name']:<20} {cp['timestamp'][:16]}{name_tag}"

    options = [(_cp_label(cp), cp["checkpoint_id"]) for cp in checkpoints]

    dropdown = widgets.Dropdown(
        options=options,
        description="Checkpoint:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "95px"},
    )
    btn_load = widgets.Button(
        description="Загрузить память",
        button_style="primary",
        icon="download",
        layout=widgets.Layout(width="170px"),
    )
    btn_fork = widgets.Button(
        description="Форк + запуск",
        button_style="info",
        icon="play",
        layout=widgets.Layout(width="155px"),
    )
    out = widgets.Output()

    def on_load(b):
        with out:
            out.clear_output()
            mem = my_agent.restore_memory(session_id, checkpoint_id=dropdown.value)
            print(f"✓ Память загружена из {dropdown.value[:8]}...\n  {mem}")

    def on_fork(b):
        with out:
            out.clear_output()
            new_sid = my_agent.fork(
                session_id,
                checkpoint_id=dropdown.value,
                description=f"fork от {dropdown.value[:8]}",
            )
            print(f"✓ Форк: {new_sid[:8]}... Запускаем агента...")
            result = my_agent.invoke(["Продолжи работу"], session_id=new_sid)
            print(f"  → {result['messages'][-1].content[:300]}")

    btn_load.on_click(on_load)
    btn_fork.on_click(on_fork)
    display(widgets.VBox([
        widgets.HTML("<b>Выбери checkpoint:</b>"),
        dropdown,
        widgets.HBox([btn_load, btn_fork]),
        out,
    ]))

── Краш-рекавери (cron-сценарий) ──
До: memory = {'calculator_last': '[calculator] 2+2 = 4', 'result_2_plus_2': '4', 'summary_general': "Calculated 2+2 = 4 and saved under key 'result_2_plus_2'."}
ℹ Сессия '1e228b56...' уже завершена (END). Восстанавливать нечего. Выбери другой checkpoint или создай форк через fork().
  → Крон логирует и завершается без ошибки. Токены не потрачены.

── Восстановление памяти из последнего checkpoint ──
Память восстановлена: {'calculator_last': '[calculator] 2+2 = 4', 'result_2_plus_2': '4'}


In [9]:
from src.agents import build_agent
from src.tools.tools import get_tools_dict, reset_memory

# Создаём агента (sessions включены автоматически через config.yaml)
my_agent = build_agent("my_agent", llm)
print(my_agent)  # покажет sessions=enabled

# --- Запуск с новой сессией ---
reset_memory()
result = my_agent.invoke(["Нарисуй график: Jan=10, Feb=20, Mar=15"])
session_id = my_agent.last_session_id
print(f"\nСессия: {session_id}")

# --- Список чекпоинтов сессии ---
checkpoints = my_agent.list_checkpoints(session_id)
print(f"\nЧекпоинтов в сессии: {len(checkpoints)}")
for cp in checkpoints:
    name_tag = f" ★ {cp['name']}" if cp.get("is_named") else ""
    print(f"  [{cp['state_name']:15}] {cp['timestamp'][:19]}{name_tag}")

MyAgent(id=MyAgent_2414715831840, states=2, sessions=enabled)
RUN 993de1cc | MyAgent_2414715831840 | started
STATE START -> work
  REQ 1 | msgs=2 in=1137 out=77
    [SYS] Ты полезный помощник для математических вычислений и визуализации данных.

Твои возможности:
- calculator: вычисление математических выражений
- ask_human: задать вопрос пользователю, если не хватает данных
- think: зафиксировать свои размышления
- memory: сохранить промежуточные результаты или прочитать сохраненное
- plot_chart: построить график (bar/line/pie) — он автоматически появится у пользователя

Алгоритм работы:
1. Проанализируй запрос пользователя
2. Если не хватает данных - используй ask_human для уточнения
3. Используй think для размышлений о подходе к решению
4. Выполни вычисления через calculator
5. Если результаты стоит визуализировать — построй график через plot_chart
6. Сохрани важные результаты через memory
7. Вызови transition(next_state="summarize") — это обязательный последний шаг


## Переход в д

## Streamlit UI

Графический интерфейс для запуска агентов и общения с ними через браузер.  
Файл: `src/ui/streamlit_ui.py`

**Возможности:**
- Выбор агента из списка и запуск с произвольным сообщением
- Лента событий в реальном времени (вызовы инструментов, ответы LLM, переходы состояний)
- Вопрос/ответ — вопросы агента появляются в интерфейсе, ответ вводится там же
- Кнопка Стоп + Перезапуск — остановить агента и запустить заново в любой момент

In [3]:
import subprocess, sys, time, socket
from pathlib import Path

_PID_FILE = Path(".streamlit.pid")
_PORT = 8501

def _port_open(port: int) -> bool:
    try:
        with socket.create_connection(("localhost", port), timeout=1):
            return True
    except OSError:
        return False

# Если Streamlit уже запущен — не запускаем второй экземпляр
if _port_open(_PORT):
    print(f"✅ Streamlit уже работает: http://localhost:{_PORT}")
    print("   Чтобы перезапустить — сначала выполните ячейку ОСТАНОВКИ ниже, затем эту снова.")
else:
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "src/ui/streamlit_ui.py",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    _PID_FILE.write_text(str(proc.pid))
    print(f"Запускаем Streamlit (PID {proc.pid}), ждём готовности", end="", flush=True)

    for _ in range(90):          # ждём до 90 секунд (тяжёлые импорты)
        time.sleep(1)
        print(".", end="", flush=True)
        if proc.poll() is not None:  # процесс упал
            print(f"\n❌ Streamlit завершился с кодом {proc.returncode}.")
            print("Запустите в терминале: streamlit run src/ui/streamlit_ui.py  — чтобы увидеть ошибку.")
            break
        if _port_open(_PORT):
            print(f"\n✅ Готово! Откройте: http://localhost:{_PORT}")
            break
    else:
        print(f"\n⚠️ Превышено время ожидания. Попробуйте вручную: http://localhost:{_PORT}")


Запускаем Streamlit (PID 12212), ждём готовности.
✅ Готово! Откройте: http://localhost:8501


In [6]:
import os, sys, signal, subprocess, socket, time
from pathlib import Path

_PID_FILE = Path(".streamlit.pid")
_PORT = 8501

def _port_open(port: int) -> bool:
    try:
        with socket.create_connection(("localhost", port), timeout=1):
            return True
    except OSError:
        return False

def _kill_pid(pid: int) -> bool:
    """Завершает процесс по PID. Возвращает True если успешно."""
    try:
        if sys.platform == "win32":
            r = subprocess.run(["taskkill", "/PID", str(pid), "/F", "/T"],
                               capture_output=True)
            return r.returncode == 0
        else:
            os.kill(pid, signal.SIGTERM)
            return True
    except (subprocess.CalledProcessError, ProcessLookupError, OSError):
        return False

def _find_pid_by_port(port: int) -> int | None:
    """Ищет PID процесса слушающего порт. Кроссплатформенно."""
    if sys.platform == "win32":
        result = subprocess.run(
            ["powershell", "-ExecutionPolicy", "Bypass", "-Command",
             f"(Get-NetTCPConnection -LocalPort {port} -ErrorAction SilentlyContinue).OwningProcess"],
            capture_output=True, text=True
        )
    else:
        result = subprocess.run(["lsof", "-ti", f"tcp:{port}"],
                                capture_output=True, text=True)
    pid_str = result.stdout.strip().splitlines()[0] if result.stdout.strip() else ""
    return int(pid_str) if pid_str.isdigit() else None

# --- Остановка ---
# Шаг 1: убиваем по PID-файлу (если есть); файл удаляем всегда (в т.ч. при битом содержимом)
if _PID_FILE.exists():
    try:
        pid = int(_PID_FILE.read_text().strip())
        _kill_pid(pid)
    except ValueError:
        pass
    finally:
        _PID_FILE.unlink(missing_ok=True)

# Шаг 2: всегда проверяем порт — убиваем что осталось
time.sleep(0.5)
if _port_open(_PORT):
    pid = _find_pid_by_port(_PORT)
    if pid:
        if _kill_pid(pid):
            print(f"Streamlit (PID {pid}) остановлен.")
        else:
            print(f"Не удалось остановить процесс {pid}. Попробуйте вручную.")
    else:
        print(f"Порт {_PORT} занят неизвестным процессом.")
else:
    print(f"Streamlit остановлен — порт {_PORT} свободен.")

Streamlit остановлен — порт 8501 свободен.
